1: Import thư viện

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from keras.models import load_model
from keras.applications.resnet50 import preprocess_input
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

2: Khởi tạo MediaPipe và các thông số

In [ ]:
import os
mp_hands = mp.solutions.hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
mp_face_mesh = mp.solutions.face_mesh.FaceMesh(max_num_faces=1, min_detection_confidence=0.7)

MODEL_PATH = r"train\vsl_cnn_pose_keras.h5"
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = r"D:\file\project\Sign-Language-Classification-v1.0\train\vsl_cnn_pose_keras.h5"

model = load_model(MODEL_PATH)

3: Import heatmap dùng chung + hàm hiển thị tiếng Việt

In [ ]:
import os
# Import create_heatmap từ module dùng chung (BR-002) — KHÔNG định nghĩa lại
import sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "src")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from src.vsl.heatmap import create_heatmap
from src.vsl.labels import VIETNAMESE_ALPHABET

def draw_vietnamese_text(image, text, position, font_size=30, color=(0, 255, 0)):
    image_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(image_pil)
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except IOError:
        print("Không tìm thấy font Arial, dùng mặc định.")
        font = ImageFont.load_default()
    draw.text(position, text, font=font, fill=color)
    return cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)

4: Chạy webcam và hiển thị ảnh trong notebook

In [ ]:
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Không thể mở camera (index 0). Thử đổi thành cv2.VideoCapture(1) nếu bạn có nhiều camera.")
else:
    print("Webcam đã bật. Đưa cả 1 bàn tay và khuôn mặt vào khung hình.")
    print("Nhấn phím 'q' trên cửa sổ OpenCV để dừng camera.")

try:
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("Không thể đọc frame.")
            break

        image = cv2.flip(image, 1)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Nhận diện tay
        hands_results = mp_hands.process(image_rgb)
        hand_landmarks = []
        if hands_results.multi_hand_landmarks:
            for hand_landmark in hands_results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(image, hand_landmark, mp.solutions.hands.HAND_CONNECTIONS)
                for landmark in hand_landmark.landmark:
                    hand_landmarks.extend([landmark.x, landmark.y, landmark.z])

        # Nhận diện mặt
        face_results = mp_face_mesh.process(image_rgb)
        face_landmarks = []
        if face_results.multi_face_landmarks:
            for face_landmark in face_results.multi_face_landmarks:
                key_points = [10, 152, 234, 454, 33, 263, 1, 61, 291, 199]
                for idx in key_points:
                    landmark = face_landmark.landmark[idx]
                    face_landmarks.extend([landmark.x, landmark.y, landmark.z])

        all_landmarks = hand_landmarks + face_landmarks
        has_hand = bool(hands_results.multi_hand_landmarks)
        has_face = bool(face_results.multi_face_landmarks)

        if all_landmarks and len(all_landmarks) == 93:
            heatmap = create_heatmap(all_landmarks)
            heatmap_input = np.expand_dims(heatmap, axis=0)
            heatmap_input = preprocess_input(heatmap_input)

            predictions = model(heatmap_input, training=False).numpy()
            predicted_class = np.argmax(predictions, axis=1)[0]
            predicted_letter = VIETNAMESE_ALPHABET[predicted_class]
            confidence = np.max(predictions) * 100

            text = f"Letter: {predicted_letter} ({confidence:.2f}%)"
            image = draw_vietnamese_text(image, text, position=(10, 30), font_size=32, color=(0, 255, 0))
        else:
            status = f"Dang tim: Tay [{'OK' if has_hand else 'X'}] | Mat [{'OK' if has_face else 'X'}]"
            cv2.putText(image, status, (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)

        cv2.imshow("Sign Language Classification (Nhan 'q' de thoat)", image)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Đã thoát theo yêu cầu.")
            break

except KeyboardInterrupt:
    print("Dừng camera.")
finally:
    cap.release()
    cv2.destroyAllWindows()
